# MAP-SAAS and additive GP models

This notebook compares three high-dimensional exact-GP alternatives: `AdditiveMapSaasSingleTaskGP`, `EnsembleMapSaasSingleTaskGP`, and `OrthogonalAdditiveGP`.

In [ ]:
import torch
from botorch.fit import fit_gpytorch_mll
from robotorchan.models import (
    AdditiveMapSaasSingleTaskGP,
    EnsembleMapSaasSingleTaskGP,
    OrthogonalAdditiveGP,
)

torch.manual_seed(0)
dtype = torch.double

## 1. Synthetic high-dimensional function

In [ ]:
n, d = 36, 10
train_X = torch.rand(n, d, dtype=dtype)
def f(X):
    return (
        torch.sin(2 * torch.pi * X[..., 0])
        + 0.7 * (X[..., 3] - 0.5) ** 2
        - 0.5 * X[..., 6]
    ).unsqueeze(-1)
train_Y = f(train_X) + 0.03 * torch.randn(n, 1, dtype=dtype)
train_X.shape, train_Y.shape

## 2. Additive MAP-SAAS

In [ ]:
additive_saas = AdditiveMapSaasSingleTaskGP(train_X, train_Y, num_taus=3)
print(additive_saas.raw_train_X.shape, additive_saas.raw_train_Y.shape)
print('supports_mll =', additive_saas.supports_mll)
fit_gpytorch_mll(additive_saas.make_mll())

## 3. Ensemble MAP-SAAS

In [ ]:
ensemble_saas = EnsembleMapSaasSingleTaskGP(train_X, train_Y, num_taus=3)
print(ensemble_saas.raw_train_X.shape, ensemble_saas.raw_train_Y.shape)
fit_gpytorch_mll(ensemble_saas.make_mll())

## 4. Orthogonal additive GP

In [ ]:
orthogonal = OrthogonalAdditiveGP(train_X, train_Y, second_order=False)
print(orthogonal.raw_train_X.shape, orthogonal.raw_train_Y.shape)
fit_gpytorch_mll(orthogonal.make_mll())

## 5. Compare posterior means on the same slice

In [ ]:
grid = torch.linspace(0, 1, 80, dtype=dtype)
X_test = torch.full((80, d), 0.5, dtype=dtype)
X_test[:, 0] = grid
models = {
    'additive MAP-SAAS': additive_saas,
    'ensemble MAP-SAAS': ensemble_saas,
    'orthogonal additive': orthogonal,
}
for name, model in models.items():
    with torch.no_grad():
        mean = model.posterior(X_test).mean.squeeze(-1)
    print(name, 'mean range =', (float(mean.min()), float(mean.max())))

## 6. Model-selection guidance

- **Additive MAP-SAAS**: useful when high-dimensional sparsity and additive structure are both plausible.
- **Ensemble MAP-SAAS**: averages multiple MAP-SAAS settings and is a practical alternative to full NUTS.
- **OrthogonalAdditiveGP**: useful when first-order or low-order additive decomposition is a substantive modeling assumption.
- Unlike fully Bayesian SAAS, these models retain the exact-GP `make_mll()` fitting workflow.